# 02. 기준선: 위도대 × 월별 비율

`MODIS ≈ 비율 × VIIRS` — 비율은 (위도대, 월)마다 학습 기간에서 구합니다.
**모델은 이 방법보다 나아야 쓸 이유가 있습니다.** 모델이 늦어져도 이 기준선만으로 조화된 달력을 만들 수 있습니다.

분할: 여러 해의 아카이브면 시간 분할(MODIS 신뢰 기간(~2024-12) 안에서 마지막 2년 시험, 그 전 해 검증. 지금 데이터로는 2022 검증, 2023–2024 시험), 7일 샘플이면 공간 분할(10° 블록 단위)입니다.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from fsml import evaluate  # 한글 폰트 설정 포함
from fsml.features import training_rows
from fsml.paths import default_grid

from fsml.baseline import RatioBaseline
from fsml.paths import PROCESSED
from fsml.splits import auto_split

df = training_rows(pd.read_parquet(default_grid()))
kind, (train, valid, test) = auto_split(df)
print(f"분할: {kind} | train {train.sum():,} / valid {valid.sum():,} / test {test.sum():,}")

In [ ]:
base = RatioBaseline().fit(df[train | valid])
df["baseline"] = base.predict(df)
print(f"전역 비율 {base.global_ratio:.3f} | 그룹별 비율 {len(base.ratios)}개")
evaluate.compare(df.loc[test, "modis_n"], {
    "기준선": df.loc[test, "baseline"],
    "VIIRS 그대로": df.loc[test, "viirs_n"],
})

In [ ]:
evaluate.plot_pred_vs_true(df.loc[test, "modis_n"], df.loc[test, "baseline"], "기준선 (시험 세트)");

## 달력 수준 정확도

달력은 칸 하나가 아니라 **지역·기간 합계**를 보여주므로, 합계 단위 오차도 확인합니다.

In [ ]:
totals = evaluate.aggregate(df[test], ["baseline"], by=["period"])
evaluate.plot_series(totals, "시험 세트 기간별 합계: MODIS 실제 vs 기준선")
pd.Series(evaluate.metrics(totals["modis_n"], totals["baseline"]))

In [ ]:
PROCESSED.mkdir(parents=True, exist_ok=True)
df[["row", "col", "period", "baseline"]].to_parquet(PROCESSED / "baseline_predictions.parquet", index=False)